In [11]:
class Tokenizer:
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.size = len(self.chars)

        self.stoi = {ch:i for i, ch in enumerate(self.chars)}
        self.itos = {i:ch for i, ch in enumerate(self.chars)}
    
    def encode(self, s):
        return [self.stoi[c] for c in s]
    def decode(self, s):
        return ''.join([self.itos[i] for i in s])
    

In [12]:
with open('STI.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

tokenizer = Tokenizer(text)

encoding = tokenizer.encode(text)

print(encoding)

decoding = tokenizer.decode(encoding)

print(decoding)

context = 5

x = []
y = []


[40, 53, 50, 1, 21, 63, 48, 53, 54, 65, 50, 48, 65, 66, 63, 50, 1, 60, 51, 1, 33, 60, 49, 50, 63, 59, 1, 25, 59, 65, 50, 63, 61, 63, 54, 64, 50, 19, 1, 21, 1, 23, 60, 58, 61, 63, 50, 53, 50, 59, 64, 54, 67, 50, 1, 25, 64, 64, 46, 70, 1, 60, 59, 1, 65, 53, 50, 1, 29, 59, 51, 60, 63, 58, 46, 65, 54, 60, 59, 1, 39, 70, 64, 65, 50, 58, 64, 1, 33, 46, 55, 60, 63, 0, 0, 40, 53, 50, 1, 46, 48, 46, 49, 50, 58, 54, 48, 1, 49, 54, 64, 48, 54, 61, 57, 54, 59, 50, 1, 60, 51, 1, 29, 59, 51, 60, 63, 58, 46, 65, 54, 60, 59, 1, 39, 70, 64, 65, 50, 58, 64, 1, 60, 48, 48, 66, 61, 54, 50, 64, 1, 46, 1, 66, 59, 54, 62, 66, 50, 1, 46, 59, 49, 1, 67, 54, 65, 46, 57, 1, 61, 60, 64, 54, 65, 54, 60, 59, 1, 68, 54, 65, 53, 54, 59, 1, 65, 53, 50, 1, 58, 60, 49, 50, 63, 59, 1, 66, 59, 54, 67, 50, 63, 64, 54, 65, 70, 1, 57, 46, 59, 49, 64, 48, 46, 61, 50, 9, 1, 43, 53, 54, 57, 50, 1, 60, 51, 65, 50, 59, 1, 48, 60, 59, 51, 66, 64, 50, 49, 1, 47, 70, 1, 57, 46, 70, 61, 50, 60, 61, 57, 50, 1, 68, 54, 65, 53, 1, 61, 6

In [13]:
import torch

# Automatically select GPU if available, otherwise use CPU
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

Using device: cuda


In [19]:

import torch

tensor_ids = torch.tensor(encoding, dtype= torch.long)


x_data = []
y_data = []
context = 64
for i in range(len(tensor_ids) - context):
    x = tensor_ids[i : i + context]
    y = tensor_ids[i + 1 : i + context + 1]

    x_data.append(x)
    y_data.append(y)

x_data = torch.stack(x_data)
y_data = torch.stack(y_data)

x_data = x_data.to(device= device)
y_data = y_data.to(device= device)



print(x_data)
print(y_data)

    


tensor([[40, 53, 50,  ..., 60, 59,  1],
        [53, 50,  1,  ..., 59,  1, 65],
        [50,  1, 21,  ...,  1, 65, 53],
        ...,
        [47, 50, 49,  ..., 47, 66, 54],
        [50, 49, 63,  ..., 66, 54, 57],
        [49, 63, 60,  ..., 54, 57, 65]], device='cuda:0')
tensor([[53, 50,  1,  ..., 59,  1, 65],
        [50,  1, 21,  ...,  1, 65, 53],
        [ 1, 21, 63,  ..., 65, 53, 50],
        ...,
        [50, 49, 63,  ..., 66, 54, 57],
        [49, 63, 60,  ..., 54, 57, 65],
        [63, 60, 48,  ..., 57, 65,  9]], device='cuda:0')


In [ ]:
import torch.nn as nn


class RNN_Type_Shi(nn.Module):
  # 1. Renamed 'embed_size' to 'vocab_size' to make it crystal clear
  def __init__(self, vocab_size, embedding_dim, hidden_dim):
    super(RNN_Type_Shi, self).__init__()

    self.vocab_size = vocab_size
    self.hidden_dim = hidden_dim
    self.embedding_dim = embedding_dim

    # 2. Embedding layer maps token IDs -> vectors (vocab_size rows, embedding_dim width)
    self.embed = nn.Embedding(
        num_embeddings=self.vocab_size, embedding_dim=self.embedding_dim
    )

    # 3. RNN processes the sequence vectors
    self.rnn = nn.RNN(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        batch_first=True, 
        num_layers= 5 ,
        dropout= 0.2
        # Keeps tensor shape as (Batch, Seq, Features)
    )

    # 4. Linear layer projects hidden dim back out to the entire vocabulary size (logits)
    self.linear = nn.Linear(in_features=self.hidden_dim, out_features=self.vocab_size)

  def forward(self, x):
    embeddings = self.embed(x)
    rnn_out, hidden = self.rnn(embeddings)
    logits = self.linear(rnn_out)
    return logits


# --- INITIALIZATION & TESTING ---
torch.cuda.empty_cache()  # Free any leftover GPU memory from previous runs

embedding_dim = 128
hidden_dim = 512
vocab_size = (
    tokenizer.size
)  # Make sure this matches your tokenizer's vocab size attribute!

# Instantiate the model correctly
llm = RNN_Type_Shi(vocab_size, embedding_dim, hidden_dim).to(device= device)

# Test a forward pass on ONLY the first 2 rows (passing all 22k rows at once causes CUDA OOM!)
with torch.no_grad():
    logits = llm(x_data[:2])
print("Test logits shape:", logits.shape)


Test logits shape: torch.Size([2, 64, 73])


In [21]:
import torch.optim as optim

# ==========================================
# 1. LOSS FUNCTION & OPTIMIZER SETUP
# ==========================================
# CrossEntropyLoss expects flat 2D logits and 1D target token IDs
criterion = nn.CrossEntropyLoss()

# Adam optimizer handles updating all 10 layers of weights via backpropagation
optimizer = optim.Adam(llm.parameters(), lr=0.0003)

# ==========================================
# 2. TRAINING LOOP (MINI-BATCHED FOR 4GB VRAM)
# ==========================================
epochs = 3000
batch_size = 512  # Sample 512 sequences per step so we don't run out of GPU memory
print(
    "Starting training loop for deep RNN on your RTX GPU..."
)

llm.train()
for epoch in range(epochs):
  # Randomly sample a mini-batch of indices
  ix = torch.randint(0, len(x_data), (batch_size,), device=device)
  x_batch = x_data[ix]
  y_batch = y_data[ix]

  # Step 1: Clear out any old accumulated gradients
  optimizer.zero_grad()

  # Step 2: Forward pass — push the mini-batch through your RNN
  logits = llm(x_batch)

  # Step 3: Flatten 3D logits and 2D targets into shapes required by CrossEntropyLoss
  b_size, seq_length, vocab_size_dim = logits.shape
  logits_flat = logits.reshape(b_size * seq_length, vocab_size_dim)
  targets_flat = y_batch.reshape(b_size * seq_length)

  # Step 4: Calculate the error penalty (loss)
  loss = criterion(logits_flat, targets_flat)

  # Step 5: Backward pass — compute backpropagation gradients across all 10 layers
  loss.backward()

  # Step 6: Optimizer step — update model weights to reduce future errors
  optimizer.step()

  # Print progress every 20 epochs to track the loss dropping
  if (epoch + 1) % 20 == 0:
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

print(
    "\nTraining complete! Your deep 10-layer RNN has successfully optimized its"
    " weights."
)

Starting training loop for deep RNN on your RTX GPU...
Epoch [20/3000], Loss: 3.0770
Epoch [40/3000], Loss: 3.0368
Epoch [60/3000], Loss: 2.8322
Epoch [80/3000], Loss: 2.5752
Epoch [100/3000], Loss: 2.3902
Epoch [120/3000], Loss: 2.2428
Epoch [140/3000], Loss: 2.1136
Epoch [160/3000], Loss: 2.0256
Epoch [180/3000], Loss: 1.9406
Epoch [200/3000], Loss: 1.8786
Epoch [220/3000], Loss: 1.7958
Epoch [240/3000], Loss: 1.7170
Epoch [260/3000], Loss: 1.6865
Epoch [280/3000], Loss: 1.6427
Epoch [300/3000], Loss: 1.5640
Epoch [320/3000], Loss: 1.5281
Epoch [340/3000], Loss: 1.4988
Epoch [360/3000], Loss: 1.4610
Epoch [380/3000], Loss: 1.4351
Epoch [400/3000], Loss: 1.3627
Epoch [420/3000], Loss: 1.3511
Epoch [440/3000], Loss: 1.3130
Epoch [460/3000], Loss: 1.2848
Epoch [480/3000], Loss: 1.2858
Epoch [500/3000], Loss: 1.2178
Epoch [520/3000], Loss: 1.2167
Epoch [540/3000], Loss: 1.1845
Epoch [560/3000], Loss: 1.1718
Epoch [580/3000], Loss: 1.1391
Epoch [600/3000], Loss: 1.1112
Epoch [620/3000], L

In [26]:
# ==========================================
# 3. INFERENCE / TEXT GENERATION
# ==========================================
def generate_text(prompt, max_new_tokens=200, temperature=0.8):
    llm.eval()  # Switch model to evaluation mode
    
    # Encode the user prompt and move it to the GPU (shape: [1, seq_len])
    input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    generated_ids = list()

    with torch.no_grad():  # Disable gradient tracking for fast inference
        for _ in range(max_new_tokens):
            # Keep only the last `context` tokens so it matches training window
            idx_cond = input_ids[:, -context:]
            
            # 1. Forward pass
            logits = llm(idx_cond)  # Shape: (1, seq_len, vocab_size)
            
            # 2. Take logits from the very last time step and scale by temperature
            next_token_logits = logits[:, -1, :] / temperature  # Shape: (1, vocab_size)
            
            # 3. Convert logits to probabilities via Softmax
            probs = torch.softmax(next_token_logits, dim=-1)
            
            # 4. Sample the next token ID from the probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # Shape: (1, 1)
            
            # 5. Append predicted token to our running sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)
            generated_ids.append(next_token.item())

    return tokenizer.decode(generated_ids)


# 👇 PUT YOUR INPUT PROMPT RIGHT HERE:
user_prompt = input("input prompt")

print("--- Generated Output ---")
print(generate_text(user_prompt, max_new_tokens=500, temperature=0.3))


--- Generated Output ---
 engineered for Online Transaction Processing: they handle thousands of concurrent users executing rapid, short-duration inserts, updates, and deletes on current operational data stored in highly normalized third-normal-form tables. However, if a business analyst attempts to execute a complex strategic query across ten years of historical sales data, joining fifteen normalized tables and aggregating hundreds of millions of rows directly on the live production database, the resulting CPU and disk


In [28]:
# ==========================================
# 4. SAVE MODEL TO A FILE (.pth)
# ==========================================
checkpoint = {
    "model_state_dict": llm.state_dict(),
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "hidden_dim": hidden_dim,
    "context": context,
    "chars": tokenizer.chars,
}

torch.save(checkpoint, "sti_rnn_model.pth")
print("✅ Model and tokenizer saved to 'sti_rnn_model.pth'!")


✅ Model and tokenizer saved to 'sti_rnn_model.pth'!


In [29]:
# ==========================================
# 5. LOAD MODEL FROM FILE (NO RETRAINING NEEDED)
# ==========================================
checkpoint = torch.load("sti_rnn_model.pth", map_location=device)

# Rebuild tokenizer & context from the saved checkpoint
tokenizer = Tokenizer("".join(checkpoint["chars"]))
context = checkpoint["context"]

# Rebuild model architecture and load trained weights
llm = RNN_Type_Shi(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
).to(device)

llm.load_state_dict(checkpoint["model_state_dict"])
llm.eval()
print("✅ Model loaded from 'sti_rnn_model.pth' and ready for inference!")


✅ Model loaded from 'sti_rnn_model.pth' and ready for inference!


C:\Users\fikrifalah\AppData\Local\Temp\ipykernel_24696\3310457923.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("sti_rnn_model.pth", map_locati